# Multi-Seed Training for Arabic ITSM Classification

Addresses **reviewer Issue #2**: single-seed experiments insufficient.

Runs 4 key configurations × 3 seeds = 12 training runs on Kaggle T4 GPU.

**Estimated time**: ~2 GPU hours total (≈10 min per run × 12 runs)

**Configurations**:
1. MARBERTv2  L1-only (seeds: 42, 123, 456)
2. MARBERTv2  L1+L2+L3 (seeds: 42, 123, 456)
3. MARBERTv2  all-5-heads (seeds: 42, 123, 456)
4. AraBERTv2  L1+L2+L3 (seeds: 42, 123, 456)

Note: seed=42 runs replicate the original experiments to confirm reproducibility.

In [1]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

fatal: destination path 'arabic-itsm-classification' already exists and is not an empty directory.
/kaggle/working/arabic-itsm-classification


In [2]:
# Link the processed dataset
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/
!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

label_encoders.pkl  test.csv  train.csv  val.csv


In [3]:
!nvidia-smi

Tue Mar 10 18:54:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Download models fully to local disk before training.
# This replaces HuggingFace lazy-streaming (which can stall at 30-40%) with
# a single complete download that all 12 training runs then load from local disk.
from huggingface_hub import snapshot_download

print('Downloading MARBERTv2 to local disk...')
MARBERT_PATH = snapshot_download(
    'UBC-NLP/MARBERTv2',
    local_dir='/kaggle/working/model_cache/marbert'
)
print('MARBERTv2 ready at:', MARBERT_PATH)

print('Downloading AraBERTv2 to local disk...')
ARABERT_PATH = snapshot_download(
    'aubmindlab/bert-base-arabertv02',
    local_dir='/kaggle/working/model_cache/arabert'
)
print('AraBERTv2 ready at:', ARABERT_PATH)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

MARBERTv2 ready at: /kaggle/working/model_cache/marbert


Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

AraBERTv2 ready at: /kaggle/working/model_cache/arabert


In [5]:
import os
from pathlib import Path

METRICS_DIR = Path('results/metrics/multi_seed')
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

# task_label matches what train.py computes:
#   --task l1         -> l1
#   --tasks l1 l2 l3  -> l1_l2_l3
#   --multi-task      -> multi_task
#
# model uses local paths (set in previous cell) to avoid HF network calls during training.
CONFIGS = [
    {'name': 'marbert_l1',     'model': MARBERT_PATH, 'flags': '--task l1',        'task_label': 'l1',        'epochs': 5},
    {'name': 'marbert_l1l2l3', 'model': MARBERT_PATH, 'flags': '--tasks l1 l2 l3', 'task_label': 'l1_l2_l3',  'epochs': 10},
    {'name': 'marbert_5heads', 'model': MARBERT_PATH, 'flags': '--multi-task',     'task_label': 'multi_task', 'epochs': 10},
    {'name': 'arabert_l1l2l3', 'model': ARABERT_PATH, 'flags': '--tasks l1 l2 l3', 'task_label': 'l1_l2_l3',  'epochs': 10},
]

print('Total runs:', len(CONFIGS) * len(SEEDS))
for cfg in CONFIGS:
    for seed in SEEDS:
        print(' ', cfg['name'], 'seed=' + str(seed))


Total runs: 12
  marbert_l1 seed=42
  marbert_l1 seed=123
  marbert_l1 seed=456
  marbert_l1l2l3 seed=42
  marbert_l1l2l3 seed=123
  marbert_l1l2l3 seed=456
  marbert_5heads seed=42
  marbert_5heads seed=123
  marbert_5heads seed=456
  arabert_l1l2l3 seed=42
  arabert_l1l2l3 seed=123
  arabert_l1l2l3 seed=456


In [6]:
import json, os, time

ipy = get_ipython()
run_log = []

for cfg in CONFIGS:
    for seed in SEEDS:
        run_key = cfg['name'] + '_seed' + str(seed)
        out_metrics_path = METRICS_DIR / (run_key + '_metrics.json')

        if out_metrics_path.exists():
            print('[SKIP]', run_key, '-- already exists')
            continue

        seed_dir = 'models/seed_' + str(seed)
        checkpoint_dir = Path(seed_dir) / ('marbert_' + cfg['task_label'] + '_best')

        parts = ['python scripts/train.py', cfg['flags'],
                 '--model', cfg['model'],
                 '--epochs', str(cfg['epochs']),
                 '--seed', str(seed),
                 '--output-dir', seed_dir,
                 '--data-dir data/processed']
        cmd = ' '.join(parts)

        print('')
        print('[RUN]', run_key)
        print('  CMD:', cmd)
        print('  Checkpoint will be:', checkpoint_dir)
        t0 = time.time()
        ipy.system(cmd)
        elapsed = time.time() - t0
        print('  Time: %.1f min' % (elapsed/60,))

        test_metrics_file = checkpoint_dir / 'test_metrics.json'
        if test_metrics_file.exists():
            with open(test_metrics_file) as fp:
                test_metrics = json.load(fp)
            result = {'config': cfg['name'], 'seed': seed, 'metrics': test_metrics}
            with open(out_metrics_path, 'w') as fp:
                json.dump(result, fp, indent=2)
            print('  Saved:', out_metrics_path)
            run_log.append({'run': run_key, 'status': 'ok', 'time_min': elapsed/60})
        else:
            print('  WARNING: test_metrics.json not found at', test_metrics_file)
            run_log.append({'run': run_key, 'status': 'no_metrics', 'time_min': elapsed/60})

print('')
print('=== Run log ===')
for r in run_log:
    print(' ', r['run'] + ':', r['status'], '(%.1f min)' % r['time_min'])



[RUN] marbert_l1_seed42
  CMD: python scripts/train.py --task l1 --model /kaggle/working/model_cache/marbert --epochs 5 --seed 42 --output-dir models/seed_42 --data-dir data/processed
  Checkpoint will be: models/seed_42/marbert_l1_best
Device: cuda | FP16: True
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Tasks: ['l1'] | Classes: {'l1': 6}
Loading weights: 100%|█| 199/199 [00:00<00:00, 3148.76it/s, Materializing param=
BertModel LOAD REPORT from: /kaggle/working/model_cache/marbert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids             

In [7]:
# Aggregate results
!python scripts/aggregate_multi_seed.py --metrics-dir results/metrics/multi_seed

Loading results from results/metrics/multi_seed...

marbert_l1: 3 seeds
  l1: 0.8894 ± 0.0023  (values: [0.8892, 0.8923, 0.8866])

marbert_l1l2l3: 3 seeds
  l1: 0.8880 ± 0.0022  (values: [0.8906, 0.8881, 0.8853])
  l2: 0.8738 ± 0.0022  (values: [0.8764, 0.871, 0.874])
  l3: 0.8066 ± 0.0023  (values: [0.8084, 0.8079, 0.8033])

marbert_5heads: 3 seeds
  l1: 0.8833 ± 0.0013  (values: [0.8851, 0.8825, 0.8822])
  l2: 0.8720 ± 0.0019  (values: [0.8717, 0.8699, 0.8745])
  l3: 0.8034 ± 0.0020  (values: [0.8029, 0.8013, 0.8061])
  priority: 0.1627 ± 0.0058  (values: [0.1709, 0.1584, 0.1589])
  sentiment: 0.2188 ± 0.0006  (values: [0.2192, 0.2192, 0.2179])

arabert_l1l2l3: 3 seeds
  l1: 0.8921 ± 0.0068  (values: [0.8843, 0.9008, 0.8912])
  l2: 0.8744 ± 0.0008  (values: [0.8749, 0.8732, 0.8751])
  l3: 0.7969 ± 0.0045  (values: [0.8027, 0.7918, 0.7961])


=== Paper-ready table (mean ± std ×100) ===
Config                              L1           L2           L3     Priority    Sentiment
---------

In [8]:
# View summary
import json
with open('results/metrics/multi_seed_summary.json') as f:
    summary = json.load(f)

for config, data in summary['summary'].items():
    print(f'\n{config} (seeds: {data["seeds"]}):')
    for task, stats in data['tasks'].items():
        print(f'  {task}: {stats["mean"]*100:.2f} ± {stats["std"]*100:.2f} (F1%)')


marbert_l1 (seeds: [123, 42, 456]):
  l1: 88.94 ± 0.23 (F1%)

marbert_l1l2l3 (seeds: [123, 42, 456]):
  l1: 88.80 ± 0.22 (F1%)
  l2: 87.38 ± 0.22 (F1%)
  l3: 80.66 ± 0.23 (F1%)

marbert_5heads (seeds: [123, 42, 456]):
  l1: 88.33 ± 0.13 (F1%)
  l2: 87.20 ± 0.19 (F1%)
  l3: 80.34 ± 0.20 (F1%)
  priority: 16.27 ± 0.58 (F1%)
  sentiment: 21.88 ± 0.06 (F1%)

arabert_l1l2l3 (seeds: [123, 42, 456]):
  l1: 89.21 ± 0.68 (F1%)
  l2: 87.44 ± 0.08 (F1%)
  l3: 79.69 ± 0.45 (F1%)


In [9]:
# Package all results for download
!zip -r multi_seed_results.zip results/metrics/multi_seed/ results/metrics/multi_seed_summary.json
print('Created multi_seed_results.zip — download from Kaggle output')

  adding: results/metrics/multi_seed/ (stored 0%)
  adding: results/metrics/multi_seed/arabert_l1l2l3_seed123_metrics.json (deflated 45%)
  adding: results/metrics/multi_seed/marbert_l1l2l3_seed456_metrics.json (deflated 45%)
  adding: results/metrics/multi_seed/marbert_l1l2l3_seed42_metrics.json (deflated 44%)
  adding: results/metrics/multi_seed/marbert_5heads_seed123_metrics.json (deflated 52%)
  adding: results/metrics/multi_seed/marbert_l1_seed456_metrics.json (deflated 19%)
  adding: results/metrics/multi_seed/marbert_5heads_seed42_metrics.json (deflated 52%)
  adding: results/metrics/multi_seed/marbert_l1_seed42_metrics.json (deflated 19%)
  adding: results/metrics/multi_seed/arabert_l1l2l3_seed456_metrics.json (deflated 44%)
  adding: results/metrics/multi_seed/marbert_5heads_seed456_metrics.json (deflated 52%)
  adding: results/metrics/multi_seed/marbert_l1l2l3_seed123_metrics.json (deflated 44%)
  adding: results/metrics/multi_seed/arabert_l1l2l3_seed42_metrics.json (deflated